ベース：白地図 CARTO Light No Labels

国境線：HDX OCHA "Syrian Arab Republic – Subnational Administrative Boundaries"
https://data.humdata.org/dataset/cod-ab-syr

syr_admin0.geojson

地域線：HDX OCHA "Syrian Arab Republic – Subnational Administrative Boundaries"
https://data.humdata.org/dataset/cod-ab-syr

syr_admin1.geojson

人口ラスタ：WorldPop "Syrian Arab Republic 100m Population 2026"
https://hub.worldpop.org/geodata/summary?id=75632

syr_worldpop_2026.tif

分析手法：Raster Masking

対象地域：
・Idleb

その他：
・GeoPandasのGovernorateポリゴンを利用し、WorldPop全国人口ラスタからIdleb州のみを抽出。
・Rasterio mask() を用いて人口ラスタを切り抜き、Idleb州専用のGeoTIFFを作成。
・抽出後、idleb_worldpop_2026.tif として保存。
・人口ラスタを正規化（Normalization）および対数変換（Log Transformation）し、人口分布を可視化。
・Folium ImageOverlay により、Idleb州の人口密度分布を地図上へ重ね合わせて表示。


In [22]:
# Core Libraries
# Raster Maskingと地図作成に必要なライブラリを読み込む

import geopandas as gpd

import rasterio

import folium

import numpy as np

import matplotlib.colors as mcolors

from rasterio.mask import mask

In [23]:
# ETL Extract
# 行政区ポリゴンと人口ラスタを読み込む

admin1 = gpd.read_file("syr_admin1.geojson")

# ETL Extract
# 人口ラスタのCRSと範囲を確認
with rasterio.open("tif_1_syria_worldpop_2026.tif") as src:
    print(src.crs)
    print(src.bounds)

EPSG:4326
BoundingBox(left=35.7166658038, bottom=32.310833540089995, right=42.37666577716, top=37.320000186719994)


In [24]:
# ETL Transform
# Idleb州だけ抽出

idleb = admin1[
    admin1["adm1_name"] == "Idleb"
]

print(idleb)

  adm1_name adm1_name1 adm1_name2 adm1_name3 adm1_pcode             adm0_name  \
9     Idleb       إدلب       None       None       SY07  Syrian Arab Republic   

                  adm0_name1 adm0_name2 adm0_name3 adm0_pcode  ...  \
9  الجمهورية العربية السورية       None       None         SY  ...   

     area_sqkm version  lang lang1 lang2 lang3 adm1_ref_name center_lat  \
9  5416.462568     v02    en    ar  None  None         Idleb  35.857697   

  center_lon                                           geometry  
9  36.570879  POLYGON ((36.75562 36.32489, 36.75385 36.3287,...  

[1 rows x 22 columns]


In [25]:
# ETL Transform
# Idleb州のgeometryを取得

idleb_geom = idleb.geometry

In [26]:
print(idleb_geom)

9    POLYGON ((36.75562 36.32489, 36.75385 36.3287,...
Name: geometry, dtype: geometry


Idleb州のPolygonだけを取り出した状態

In [27]:
# ETL Transform
# Idleb州ポリゴンで人口ラスタを切り抜く

with rasterio.open("tif_1_syria_worldpop_2026.tif") as src:
    out_image, out_transform = mask(
        src,
        idleb_geom,
        crop=True
    )

print(out_image.shape)
print(out_transform)

(1, 1155, 1266)
| 0.00, 0.00, 36.15|
| 0.00,-0.00, 36.34|
| 0.00, 0.00, 1.00|


1 = バンド数、1155 = 行数 (height)、1266   = 列数 (width)

36.15 36.34 切り抜いたラスタの新しい座標

In [28]:
# ETL Transform
# 切り抜いたラスタを保存するためのメタデータを更新

with rasterio.open("tif_1_syria_worldpop_2026.tif") as src:
    out_meta = src.meta.copy()

out_meta.update({
    "driver": "GTiff",
    "height": out_image.shape[1],
    "width": out_image.shape[2],
    "transform": out_transform
})

print(out_meta)

{'driver': 'GTiff', 'dtype': 'float32', 'nodata': -99999.0, 'width': 1266, 'height': 1155, 'count': 1, 'crs': CRS.from_wkt('GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]'), 'transform': Affine(0.00083333333, 0.0, 36.14749913541,
       0.0, -0.00083333333, 36.33916685731)}


In [29]:
# ETL Load
# Idleb州だけの人口ラスタをGeoTIFFとして保存

with rasterio.open("tif_2_idleb_worldpop_2026.tif", "w", **out_meta) as dest:
    dest.write(out_image)

WorldPop Syria rasterからIdleb governorateのみを抽出し、idleb_worldpop_2026.tif　を作成した

In [30]:
# Idleb周辺へズームした白地図ベース

m = folium.Map(
    location=[35.9, 36.6],
    zoom_start=9,
    tiles='https://{s}.basemaps.cartocdn.com/light_nolabels/{z}/{x}/{y}{r}.png',
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a>'
)

In [31]:
# ETL Extract
# Idleb専用GeoTIFFを読み込む

with rasterio.open("tif_2_idleb_worldpop_2026.tif") as src:
    idleb_array = src.read(1).astype("float32")
    idleb_bounds = [[src.bounds.bottom, src.bounds.left], [src.bounds.top, src.bounds.right]]

print(idleb_array.shape)
print(idleb_bounds)

(1155, 1266)
[[35.37666686116, 36.14749913541], [36.33916685731, 37.20249913119]]


In [32]:
# ETL Transform
# 比較確認のため、人口ラスタを通常の0〜1へ変換する

idleb_array[idleb_array < 0] = np.nan

idleb_norm = (idleb_array - np.nanmin(idleb_array)) / (
    np.nanmax(idleb_array) - np.nanmin(idleb_array)
)

print(np.nanmin(idleb_norm))
print(np.nanmax(idleb_norm))

0.0
1.0


In [33]:
# ETL Transform
# 深いグリーンのカラーマップを作成

green_cmap = mcolors.LinearSegmentedColormap.from_list(
    "deep_green",
    [
        (0.0, "#ffffff"),
        (0.25, "#d9f0d3"),
        (0.50, "#1d5c37"),
        (0.75, "#00441b"),
        (1.0, "#01240f")
    ]
)

In [34]:
# 以下、色味調整のための確認とlog調整

print(np.nanmin(idleb_array))
print(np.nanmax(idleb_array))

0.0
463.28693


In [35]:
# ETL Transform
# Log変換で人口分布を見えやすくする
# Log変換後の値をFolium表示用に0〜1へ変換する

idleb_log = np.log1p(idleb_array)

idleb_log_norm = (idleb_log - np.nanmin(idleb_log)) / (
    np.nanmax(idleb_log) - np.nanmin(idleb_log)
)

print(np.nanmin(idleb_log_norm))
print(np.nanmax(idleb_log_norm))

0.0
1.0


In [36]:
# ETL Load
# Log変換後のIdleb人口ラスタを白地図へ重ねる

folium.raster_layers.ImageOverlay(
    image=idleb_log_norm,
    bounds=idleb_bounds,
    colormap=lambda x: green_cmap(x),
    opacity=0.85,
    name="Idleb Population Raster"
).add_to(m)

In [37]:
# ETL Load
# Idlebラベルを地図中央に追加

folium.Marker(
    location=[35.9, 36.6],
    icon=folium.DivIcon(
        html='''
        <div style="
            font-size: 20pt;
            color: gray;
            font-weight: bold;
            white-space: nowrap;
            text-align: center;
            width: 120px;
            margin-left: -60px;
            text-shadow: 0 0 4px white;
        ">
        Idleb
        </div>
        '''
    )
).add_to(m)

In [38]:
# ETL Transform
# Folium表示用に必要な列だけ残す

idleb_map = idleb[["adm1_name", "geometry"]]

In [39]:
# ETL Load
# Idleb州境を地図へ追加

folium.GeoJson(
    idleb_map,
    style_function=lambda x: {
        "color": "#444444",
        "weight": 2,
        "fillOpacity": 0
    }
).add_to(m)

In [40]:
title_html = '''
<div style="
position: fixed;
top: 20px;
left: 20px;
width: 420px;
height: 210px;
background-color: rgba(255,255,255,0.90);
color: black;
z-index: 9000;
padding: 15px;
border: 1px solid #999;
border-radius: 8px;
font-size: 13px;
">

<b style="font-size:18px;">
Idleb Governorate
</b><br>

<span style="color:#00441b;">
Population Density (WorldPop 2026)
</span>

<hr>

Population raster extracted from the national
WorldPop Syria dataset using raster masking.

Only the Idleb governorate polygon was retained
and visualised as an independent GeoTIFF layer.

<br><br>

Source:
WorldPop 2026

</div>
'''

In [41]:
# ETL Load
# タイトル・説明ボックスを地図へ追加

m.get_root().html.add_child(
    folium.Element(title_html)
)

In [42]:
# ETL Load
# HTMLマップとして保存

m.save("08_syria_Idleb_mask.html")